In [7]:
import torch
from transformers import GPT2LMHeadModel, DataCollatorForLanguageModeling, TrainingArguments, Trainer, set_seed
import pandas as pd
import numpy as np
from datasets import Dataset, load_from_disk, concatenate_datasets
from transformers import GPT2TokenizerFast
from pathlib import Path
import json

In [8]:
seed = 43
set_seed(seed)
np.random.seed(seed)

In [9]:
out_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/')

tokenizer = GPT2TokenizerFast.from_pretrained(out_dir / 'tokenizer')

with open(out_dir / 'token_spec.json', 'r', encoding='utf-8') as f:
    token_spec = json.load(f)
    
ds_user_behavior = load_from_disk(out_dir / 'data' / 'behavior')
ds_meta_title = load_from_disk(out_dir / 'data' / 'meta_title')
ds_meta_genre = load_from_disk(out_dir / 'data' / 'meta_genre')
ds_meta_year  = load_from_disk(out_dir / 'data' / 'meta_year')

def sample_len_stats(ds, n=2000):
    m = min(n, len(ds))
    lens = [len(ds[i]['input_ids']) for i in range(m)]
    return {
        'min': int(np.min(lens)),
        'median': int(np.median(lens)),
        'p95': int(np.percentile(lens, 95)),
        'max': int(np.max(lens)),
    }

print('behavior:', sample_len_stats(ds_user_behavior))
print('meta_title:', sample_len_stats(ds_meta_title))
print('meta_genre:', sample_len_stats(ds_meta_genre))
print('meta_year:', sample_len_stats(ds_meta_year))

ratio_behavior = 0.5

ds_meta_all = concatenate_datasets([ds_meta_title, ds_meta_genre, ds_meta_year]).shuffle(seed=seed)
ds_user_behavior = ds_user_behavior.shuffle(seed=seed)

max_total = int(min(len(ds_user_behavior) / ratio_behavior, len(ds_meta_all) / (1 - ratio_behavior)))

n_user_behavior = int(max_total * ratio_behavior)
n_meta_all = int(max_total * (1 - ratio_behavior))

ds_user_behavior_selected = ds_user_behavior.select(range(n_user_behavior))
ds_meta_all_selected = ds_meta_all.select(range(n_meta_all))

ds_cpt = concatenate_datasets([ds_user_behavior_selected, ds_meta_all_selected]).shuffle(seed=seed)

print('ds_cpt size:', len(ds_cpt), 'behavior:', n_user_behavior, 'meta:', n_meta_all)

split = ds_cpt.train_test_split(test_size=0.02, seed=seed)
ds_train = split['train']
ds_val = split['test']

print('train:', len(ds_train), 'val:', len(ds_val))

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


behavior: {'min': 152, 'median': 728, 'p95': 1008, 'max': 1008}
meta_title: {'min': 15, 'median': 17, 'p95': 22, 'max': 33}
meta_genre: {'min': 17, 'median': 20, 'p95': 25, 'max': 33}
meta_year: {'min': 16, 'median': 16, 'p95': 16, 'max': 16}
ds_cpt size: 12080 behavior: 6040 meta: 6040
train: 11838 val: 242


In [10]:
base_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2_v1')
start_ckpt = base_dir / 'checkpoint-1400'          
new_dir = base_dir.parent / 'cpt_gpt2_v1_plus'     

tokenizer = GPT2TokenizerFast.from_pretrained(str(base_dir))
model = GPT2LMHeadModel.from_pretrained(str(start_ckpt))
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [11]:
args = TrainingArguments(
    output_dir=str(new_dir),
    overwrite_output_dir=True,
    num_train_epochs=4,
    learning_rate=5e-5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8, 
    eval_strategy='steps',
    save_strategy='steps',
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    fp16=True,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    
    
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model(str(new_dir))
tokenizer.save_pretrained(str(new_dir))

Step,Training Loss,Validation Loss
200,1.718000,1.517150
400,1.538700,1.363279
600,1.440500,1.278040
800,1.366700,1.227407
1000,1.322200,1.196891
1200,1.294200,1.176541
1400,1.273900,1.168383


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


('..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1_plus\\tokenizer_config.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1_plus\\special_tokens_map.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1_plus\\vocab.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1_plus\\merges.txt',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1_plus\\added_tokens.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1_plus\\tokenizer.json')

In [12]:
base_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2_v1_plus')
best_dir = base_dir / 'checkpoint-1400'

tokenizer = GPT2TokenizerFast.from_pretrained(str(base_dir))
model = GPT2LMHeadModel.from_pretrained(str(best_dir))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

model.config.eos_token_id = tokenizer.eos_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [13]:
def generate_from_prompt(prompt, max_new_tokens=60, sample=False):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens,
            do_sample=sample,
            top_p=0.9,
            temperature=0.25,
            pad_token_id=model.config.pad_token_id,
            eos_token_id=model.config.eos_token_id,
            bos_token_id=model.config.bos_token_id
        )
        

    return tokenizer.decode(out[0])

In [17]:
n = 10

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'title' in true:
        
        colon_pos = title_tokens.index(':')
        prompt_tokens = title_tokens[:colon_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><title>Movie <sid_0_1949><sid_1_308><sid_2_33><sid_3_141><sid_4_1> has title: Going My Way</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_1949><sid_1_308><sid_2_33><sid_3_141><sid_4_1> has title: How to Train Your Dragon</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_230><sid_1_655><sid_2_337><sid_3_110><sid_4_77> has title: Cape Fear</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_230><sid_1_655><sid_2_337><sid_3_110><sid_4_77> has title: Love Story, The</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_696><sid_1_976><sid_2_355><sid_3_5><sid_4_101> has title: Lolita</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_696><sid_1_976><sid_2_355><sid_3_5><sid_4_101> has title: The Last Summer</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_38><sid_1_330><sid_2_337><

In [28]:
n = 10

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'genres' in true:
        
        colon_pos = title_tokens.index(':')
        prompt_tokens = title_tokens[:colon_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><genres>The genres in movie <sid_0_540><sid_1_650><sid_2_479><sid_3_26><sid_4_67> are:Horror,Thriller</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_540><sid_1_650><sid_2_479><sid_3_26><sid_4_67> are:Comedy</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in movie <sid_0_540><sid_1_562><sid_2_464><sid_3_205><sid_4_90> are:Romance</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_540><sid_1_562><sid_2_464><sid_3_205><sid_4_90> are:Comedy</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in movie <sid_0_739><sid_1_271><sid_2_405><sid_3_108><sid_4_18> are:Comedy</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_739><sid_1_271><sid_2_405><sid_3_108><sid_4_18> are:Comedy</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in mov

In [22]:
n = 15

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'released' in true:

        in_pos = title_tokens.index('Ġin')
        prompt_tokens = title_tokens[:in_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><year>The movie <sid_0_1277><sid_1_141><sid_2_2><sid_3_204><sid_4_20> was released in 1995</year><eos>
The GPT2 answer: <bos><year>The movie <sid_0_1277><sid_1_141><sid_2_2><sid_3_204><sid_4_20> was released in 1997</year><eos>
--------------------------------------------------
The original promt: <bos><year>The movie <sid_0_405><sid_1_302><sid_2_222><sid_3_255><sid_4_56> was released in 1984</year><eos>
The GPT2 answer: <bos><year>The movie <sid_0_405><sid_1_302><sid_2_222><sid_3_255><sid_4_56> was released in 1994</year><eos>
--------------------------------------------------
The original promt: <bos><year>The movie <sid_0_540><sid_1_459><sid_2_222><sid_3_112><sid_4_96> was released in 1984</year><eos>
The GPT2 answer: <bos><year>The movie <sid_0_540><sid_1_459><sid_2_222><sid_3_112><sid_4_96> was released in 1997</year><eos>
--------------------------------------------------
The original promt: <bos><year>The movie <sid_0_696><sid_1_141><sid_2_17><sid_3_85><